In [ ]:
import mlflow
from sklearn.metrics import roc_auc_score, roc_curve
from mlt.models.metrics import save_metric_curves
from mlt.data.preprocessing import get_data
from mlt.config import PROJECT_ROOT, get_logger, INTERIM_DATA_DIR
from mlflow.tracking import MlflowClient
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import cross_val_predict
from IPython.display import Image, display

logger = get_logger(__name__)

URI = "sqlite:///" + str(PROJECT_ROOT) + "/mlflow.db"

mlflow.set_tracking_uri(URI)
mlflow.set_registry_uri(URI)

client = MlflowClient(tracking_uri=URI, registry_uri=URI)

In [ ]:
# if this doesnt work, restart kernel or run pipeline
# newest version knn
versions_knn = client.search_model_versions("name='best_knn_model'")
latest_knn = max(versions_knn, key=lambda v: int(v.version))

print(f"laod model: customer-churn-knn v{latest_knn.version}")

pipe_knn = mlflow.sklearn.load_model(latest_knn.source)

In [ ]:
# newest version baseline
versions_baseline = client.search_model_versions("name='customer-churn-baseline'")
latest_baseline = max(versions_baseline, key=lambda v: int(v.version))

print(f"laod model: customer-churn-baseline v{latest_baseline.version}")

pipe_baseline = mlflow.sklearn.load_model(latest_baseline.source)

In [ ]:
# get data
X, y = get_data(logger=logger, INTERIM_DATA_DIR=INTERIM_DATA_DIR)

In [ ]:
# VALIDATION best knn
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_score = cross_val_predict(pipe_knn, X, y, cv=cv, method="predict_proba")[:, 1]

fpr, tpr, thresholds = roc_curve(y, y_score)
auc = roc_auc_score(y, y_score)
save_metric_curves(
    fpr=fpr, tpr=tpr, auc=auc, reports_dir=str(PROJECT_ROOT) + "/figures/model/"
)

In [ ]:
# VALIDATION baseline
y_score = cross_val_predict(pipe_baseline, X, y, cv=cv, method="predict_proba")[:, 1]

fpr, tpr, thresholds = roc_curve(y, y_score)
auc = roc_auc_score(y, y_score)
save_metric_curves(
    fpr=fpr, tpr=tpr, auc=auc, reports_dir=str(PROJECT_ROOT) + "/figures/baseline/"
)

    knn evaluated on hpo with random search for 50 <= k => 250, distance or uniform weights on 50 trials
    baseline evaluated on 1500 iterations, weight balanced

In [ ]:
display(Image(str(PROJECT_ROOT) + "/figures/model_comparison/comparison.png"))

In [ ]:
print(pipe_knn[-1])